In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="imTak/korean-audio-text-develop", 
    repo_type="dataset", local_dir="./korean-audio-text-develop", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|██████████| 7/7 [00:03<00:00,  2.23it/s]


'/home/ubuntu/korean-audio-text-develop'

In [3]:
files = glob('korean-audio-text-develop/*/*.parquet')
len(files)

7

In [4]:
df = pd.read_parquet(files[0])
df.head()

,text,name
0,거의 40만 이상 되죠,{'bytes': b'RIFFt\xd8\x04\x00WAVEfmt \x10\x00\...
1,이때 오른쪽 아래에 초대 수락을 누르게 되면,{'bytes': b'RIFF\x80\xef\x07\x00WAVEfmt \x10\x...
2,Creat and add를 눌러 줍니다,{'bytes': b'RIFF\xa4\x07\x06\x00WAVEfmt \x10\x...
3,이게 일단 세계가 뭔지 아시는 분들은 쉽다 할 텐데,{'bytes': b'RIFF$V\n\x00WAVEfmt \x10\x00\x00\x...
4,int를 담을 수 있는 num이라는 상자라는 뜻인데,{'bytes': b'RIFF`\x00\t\x00WAVEfmt \x10\x00\x0...


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['name'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 822/822 [00:34<00:00, 23.65it/s]


In [7]:
len(data)

5139

In [8]:
with open('korean-audio-text-develop.json', 'w') as fopen:
    json.dump(data, fopen)

In [ ]:
audio_files = [d['audio_filename'] for d in data]

with open('Japanese-Eroge-Voice-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)